Classification Fine-Tuning (text messages --> Spam / Not Spam)

In [6]:
import urllib.request
import zipfile
import os 
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_dataset(url,zip_path,extracted_path,data_file_path):
    if data_file_path.exists():
        print(f'{data_file_path} already exists. Skipping download and extraction.')
        return
    with urllib.request.urlopen(url) as response : ## downloads the file
        with open(zip_path,"wb") as out_file : 
            out_file.write(response.read())
    with zipfile.ZipFile(zip_path,"r") as zip_ref : 
        zip_ref.extractall(extracted_path)
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path , data_file_path)
    print(f"File Downloaded and saved as {data_file_path}")

In [7]:
download_and_unzip_dataset(url,zip_path,extracted_path,data_file_path)

sms_spam_collection\SMSSpamCollection.tsv already exists. Skipping download and extraction.


In [8]:
import pandas as pd 
df = pd.read_csv("sms_spam_collection/SMSSpamCollection.tsv" ,sep="\t",header=None,names=["Label" , "Text"])
print(df["Label"].value_counts())
df.head()

Label
ham     4825
spam     747
Name: count, dtype: int64


,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Dataset Preparation

In [9]:
## Balancing the dataset : 747 spam , 4825 ham
def balance_dataset(df) : 
    num_spam = df[df["Label"]=="spam"].shape[0]
    ham_subset = df[df["Label"]=="ham"].sample(num_spam,random_state=123)
    balanced = pd.concat([ham_subset , df[df["Label"]=="spam"]])
    return balanced

balanced_df = balance_dataset(df) ## balanced dataset 
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


In [10]:
balanced_df["Label"] = balanced_df["Label"].map({"ham":0,"spam":1})

In [11]:
## splitting the dataset 
def split_dataset(df,train_frac,val_frac) : 
    df = df.sample(
        frac=1 , random_state=123
    ).reset_index(drop=True)
    train_end = int(len(df)*train_frac)
    validation_end = train_end + int(len(df) * val_frac)
    train_df = df[:train_end]
    val_df = df[train_end:validation_end]
    test_df = df[validation_end:]
    return train_df , val_df , test_df
train_df , val_df , test_df = split_dataset(balanced_df , 0.7,0.1)

In [12]:
train_df.to_csv("train.csv" , index=None)
val_df.to_csv("val.csv" , index=None)
test_df.to_csv("test.csv" , index=None)

In [13]:
import torch 
from torch.utils.data import Dataset
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

class SpamDataset(Dataset):
    def __init__(self,csv_file,tokenizer,max_length=None , pad_token_id=50256) : 
        self.data = pd.read_csv(csv_file)
        self.texts_encoded = [tokenizer.encode(text) for text in self.data["Text"]]
        if max_length is None : 
            self.max_length = self._longest_encoded_text()
        else : 
            self.max_length = max_length 
            self.texts_encoded = [
                text_encoded[:self.max_length] for text_encoded in self.texts_encoded ## truncate text if it is longer than the specified max length 
            ]
        self.texts_encoded = [ ## add padding to texts to mach the longest text in the dataset
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text)) for encoded_text in self.texts_encoded
        ]
        
    def __getitem__(self, index):
        encoded = self.texts_encoded[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded,dtype=torch.long),
            torch.tensor(label,dtype=torch.long)
        )
    def __len__(self):
        return len(self.data)
    def _longest_encoded_text(self) : 
        max_length = 0 
        for encoded_text in self.texts_encoded : 
            encoded_length = len(encoded_text)
            if encoded_length > max_length : 
                max_length = encoded_length
        return max_length

In [14]:
train_dataset = SpamDataset("train.csv" , tokenizer)

In [15]:
print(train_dataset.max_length)

120


In [16]:
val_dataset = SpamDataset("val.csv" , tokenizer,max_length=train_dataset.max_length)
test_dataset = SpamDataset("test.csv" , tokenizer,max_length=train_dataset.max_length)
print(val_dataset.max_length)
print(test_dataset.max_length)

120
120


In [17]:
from torch.utils.data import DataLoader
num_workers = 0
batch_size = 8
torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

In [18]:
for input_batch , target_batch in train_loader : 
    pass
print(f"Input batch shape  : {input_batch.shape}")
print(f"Target batch shape : {target_batch.shape}")

Input batch shape  : torch.Size([8, 120])
Target batch shape : torch.Size([8])


In [19]:
print(f"{len(train_loader)} : training batches")
print(f"{len(val_loader)} : validation batches")
print(f"{len(test_loader)} : test batches")

130 : training batches
19 : validation batches
38 : test batches


Model Preparation

In [34]:
chosen_model = "gpt2_small (124M)"
input_prompt = "Every effort moves you"
base_config = {
    "vocab_size":50257,
    "context_length":1024,
    "drop_rate":0.0,
    "qkv_bias":True
}

models_config = {
    "gpt2_small (124M)" : {"emb_dim":768 , "n_layers":12 ,"n_heads":12} , 
    "gpt2_medium (355M)" : {"emb_dim":1024 , "n_layers":24 ,"n_heads":16} , 
    "gpt2_large (774M)" : {"emb_dim":1280 , "n_layers":36,"n_heads":20} , 
    "gpt2_xl (1558M)" : {"emb_dim":1600 , "n_layers":48 ,"n_heads":25} , 
}
base_config.update(models_config[chosen_model])
print(base_config)


{'vocab_size': 50257, 'context_length': 1024, 'drop_rate': 0.0, 'qkv_bias': True, 'emb_dim': 768, 'n_layers': 12, 'n_heads': 12}


In [25]:
from gpt_download import download_and_load_gpt2
from modules import  GPTModel , load_weights_into_gpt
model_size = chosen_model.split(" ")[-1].lstrip("(").rstrip(")")
print(f"model size : ",model_size)
settings , params = download_and_load_gpt2(
    model_size=model_size,
    models_dir="gpt2"
)

model size :  124M
File already exists and is up-to-date: gpt2\124M\checkpoint
File already exists and is up-to-date: gpt2\124M\encoder.json
File already exists and is up-to-date: gpt2\124M\hparams.json
File already exists and is up-to-date: gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2\124M\model.ckpt.index
File already exists and is up-to-date: gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: gpt2\124M\vocab.bpe


In [27]:
gpt = GPTModel(base_config)
load_weights_into_gpt(gpt,params)
gpt.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (MHattention): MultiHeadAttention(
        (w_q): Linear(in_features=768, out_features=768, bias=True)
        (w_k): Linear(in_features=768, out_features=768, bias=True)
        (w_v): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (layer_norm_1): LayerNorm()
      (layer_norm_2): LayerNorm()
      (ffn): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (MHattention): MultiHeadAttention(
        (w_q): Linear(in

In [57]:
import modules
import importlib
importlib.reload(modules)
from modules import generate , text_to_token_ids , ids_token_to_text , generate_next_token

In [55]:
ids = generate(
    model=gpt,
    idx = text_to_token_ids(input_prompt,tokenizer),
    context_size=base_config["context_length"],
    max_new_tokens=20,
    temp=1.4,
    top_k=15
)

In [56]:
print(f"Model output : \n{ids_token_to_text(ids,tokenizer)}")

Model output : 
Every effort moves you towards the ultimate goal.

You need to think of the best way to help make your work


In [60]:
""" 
    before fine-tunning the model , let's test its capacity to follow instructions
    ==> the result shows that the model is struggling with the input prompt it was given because it lacks the ability to understand and 
    follow instructions , which is done via fine-tunning as we are going to do next.
"""
text_2 = (
"Is the following text 'spam'? Answer with 'yes' or 'no':"
" 'You are a winner you have been specially"
" selected to receive $1000 cash or a $2000 award.'"
)
token_ids = generate_next_token(
model=gpt,
idx=text_to_token_ids(text_2, tokenizer),
max_new_tokens=50,
context_size=base_config["context_length"]
)
print(ids_token_to_text(token_ids, tokenizer))

Is the following text 'spam'? Answer with 'yes' or 'no': 'You are a winner you have been specially selected to receive $1000 cash or a $2000 award.'

The following text 'spam'? Answer with 'yes' or 'no': 'You are a winner you have been specially selected to receive $1000 cash or a $2000 award.'

The following text 'spam'? Answer with


In [61]:
print(gpt.out_head)

Linear(in_features=768, out_features=50257, bias=False)


In [66]:
""" 
    for our classification task we need the output to be the probabilities between two classes 0:ham , 1:spam
    to do that we need to map the output in the out_head layer from dim=768 to dim=2 (instead of 50257)
    this is called Fine-tuning selected since we are fine-tuning only the last layer (near the output)
    in order to do this we need to "freeze" the model : make all layers non-trainable
"""
n_classes = 2
torch.manual_seed(123)
gpt.out_head = torch.nn.Linear( ## ==> requires_grad = True ====> this layer is trainable
    in_features=base_config["emb_dim"],
    out_features=n_classes
)
print(gpt.out_head)

Linear(in_features=768, out_features=2, bias=True)


In [69]:
## Last transformer blocks and the final normalization layer are also trainable (this may add more efficiency to our model)

for param in gpt.trf_blocks[-1].parameters() : 
    param.requires_grad = True
for param in gpt.final_norm.parameters() : 
    param.requires_grad = True